# 🕵️ Analyse du Data Drift (Evidently AI)

Ce notebook permet d'analyser la dérive des données entre le dataset d'entraînement (**Reference**) et les prédictions effectuées en production (**Current**).

## 🛠️ Configuration

In [19]:
import pandas as pd
import sqlite3
import os
import evidently
from evidently import Report
from evidently.presets import DataDriftPreset

DB_PATH = "../data/database.sqlite"

if not os.path.exists(DB_PATH):
    DB_PATH = "data/database_lite.sqlite" # Fallback

print("✅ Imports rechargés et configuration prête.")

✅ Imports rechargés et configuration prête.


## 📥 Chargement des Données

In [20]:
conn = sqlite3.connect(DB_PATH)

# 1. Référence (Données d'entraînement)
reference_df = pd.read_sql_query("SELECT * FROM clients LIMIT 1000", conn)

# 2. Actuel (Logs de production)
current_df = pd.read_sql_query("SELECT * FROM prediction_logs", conn)

conn.close()

print(f"Taille Reference: {len(reference_df)}")
print(f"Taille Current: {len(current_df)}")

Taille Reference: 1000
Taille Current: 0


## 🧪 Analyse de la Dérive (Data Drift)

On compare les colonnes communes présentes dans les logs.

In [21]:
# 1. PRÉPARATION
monitored_features = [
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "PAYMENT_RATE", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "AMT_ANNUITY", "AMT_CREDIT", "AMT_INCOME_TOTAL"
]

ref = reference_df[monitored_features].copy().reset_index(drop=True)
cur = current_df[monitored_features].copy().reset_index(drop=True)

# 2. GÉNÉRATION
report = Report(metrics=[
    DataDriftPreset(drift_share=0.3)
])

my_report = report.run(reference_data=ref, current_data=cur)

# 3. SAUVEGARDE (Pour garder une trace physique)
my_report.save_html("drift_report.html")

# 4. AFFICHAGE (Méthode officielle : on appelle juste l'objet)
my_report

/Users/daminou/miniconda3/envs/credit-scoring-app/lib/python3.10/site-packages/evidently/legacy/calculations/data_drift.py:462: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



ZeroDivisionError: division by zero